In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/dataranch/supermarket-sales-prediction-xgboost-fastai/src/small_bench/checkpoints/pre_cell_10.pickle

In [3]:
%%RecordEvent
%%time
### cell 10 ###

import os
# Create config files once
os.makedirs(PARAM_DIR, exist_ok=True)
for fname in ('cats.txt', 'conts.txt', 'cols_to_delete.txt'):
    open(os.path.join(PARAM_DIR, fname), 'a').close()

# Initialize targets
target = ''
target_str = ''
targets = []

# Iterate through potential target columns (continuous)
for col in reversed(df.columns[:-1]):
    # Try numeric conversion; skip if all NaN
    col_numeric = pd.to_numeric(df[col], errors='coerce')
    if not col_numeric.notna().any():
        continue
    df[col] = col_numeric
    target = col
    target_str = col.replace('/', '-')
    print(f'Target Variable: {target}')

    # Drop duplicates and optionally shuffle
    df.drop_duplicates(inplace=True)
    if SHUFFLE_DATA:
        df = df.sample(frac=1).reset_index(drop=True)

    # Cast boolean columns to uint8
    bool_cols = df.select_dtypes(include='bool').columns
    if bool_cols.any():
        df[bool_cols] = df[bool_cols].astype('uint8')

    # Drop unwanted columns
    with open(os.path.join(PARAM_DIR, 'cols_to_delete.txt')) as f:
        cols_to_delete = f.read().splitlines()
    df.drop(columns=cols_to_delete, errors='ignore', inplace=True)

    # Fill missing values
    df.fillna(0, inplace=True)

    # Auto-detect categorical vs continuous
    nunique = df.nunique()
    counts = df.count()
    ratio = nunique / counts
    cats = ratio[ratio < 0.05].index.tolist()
    conts = ratio[ratio >= 0.05].index.tolist()
    if target in cats: cats.remove(target)
    if target in conts: conts.remove(target)

    # Ensure target is numeric
    df[target] = pd.to_numeric(df[target], errors='coerce')

    # Override with external variable files if required
    if VARIABLE_FILES:
        with open(os.path.join(PARAM_DIR, 'cats.txt')) as f:
            cats = f.read().splitlines()
        with open(os.path.join(PARAM_DIR, 'conts.txt')) as f:
            conts = f.read().splitlines()

    # Convert continuous variables
    for var in conts:
        try:
            df[var] = pd.to_numeric(df[var], errors='coerce')
        except Exception:
            print(f'Could not convert {var} to float.')

    # Experimental breakpoint logic
    if ENABLE_BREAKPOINT:
        cont_list = conts.copy()
        for var in conts:
            try:
                df[var] = pd.to_numeric(df[var], errors='coerce')
            except Exception:
                print(f'Could not convert {var} to float.')
                cont_list.remove(var)
                if CONVERT_TO_CAT:
                    cats.append(var)


Target Variable: Discount


Target Variable: Quantity
Target Variable: Sales
Target Variable: Postal Code
Target Variable: City
CPU times: user 466 ms, sys: 40.1 ms, total: 506 ms
Wall time: 504 ms


In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/notebooks/dataranch/supermarket-sales-prediction-xgboost-fastai/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_10_try_0.pickle